# 20 — serving_fornecedor_perfil

## Purpose
Build the `serving_fornecedor_perfil` table — a consumer-aligned data product
for the supplier dashboard. One row per supplier with aggregated contract and
sanction metrics, identifiable only via `cnpj_token` (ADR-005).

## Grain
One row per supplier (cnpj_normalized → cnpj_token), current version only.

## Input
- `local/data/gold/dim_fornecedor.parquet`
- `local/data/gold/fato_contratos.parquet`
- `local/data/gold/fato_sancoes.parquet`

## Output
- `local/data/serving/serving_fornecedor_perfil.parquet`

## Steps
1. Imports, configuration and CNPJ salt validation
2. Load Gold sources as lazy views
3. Compute cnpj_token mapping (Python HMAC-SHA256 + salt)
4. Aggregate contract metrics per supplier
5. Aggregate sanction metrics per supplier
6. Build final serving table
7. Write to Parquet
8. Validation

## Architecture Notes
- **cnpj_token:** HMAC-SHA256(cnpj_normalized, salt) — ADR-005 Privacy by Design.
  cnpj_normalized is never exposed in the output. The token is computed in Python
  because DuckDB has no native HMAC function.
- **Salt source:** os.getenv('CNPJ_SALT') from local .env file.
  In Databricks: dbutils.secrets.get(scope='fornecedor360-secrets', key='cnpj-salt').
  If CNPJ_SALT is not set, the notebook FAILS explicitly — never silently.
- **is_current = True filter:** only the current version of each supplier is included.
  Historical versions (SCD2) are not relevant for the dashboard consumer.
- **LEFT JOIN for metrics:** suppliers with no contracts or no sanctions get 0/NULL,
  not dropped. Every supplier in dim_fornecedor gets a row in serving.
- Idempotent — safe to re-run (overwrites output file).


## Step 1 — Imports, configuration and CNPJ salt validation

In [ ]:
import sys
import duckdb
import hmac
import hashlib
import os
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

# --- Resolve project root ---
def _find_root(start: Path) -> Path:
    current = start if start.is_dir() else start.parent
    for candidate in [current, *current.parents]:
        if (candidate / "utils" / "paths.py").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return start

try:
    _seed = Path(__vsc_ipynb_file__).resolve()
except NameError:
    _seed = Path.cwd().resolve()

PROJECT_ROOT = _find_root(_seed)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import gold_path, serving_path, ensure_dir

# --- Paths ---
DIM_FORNECEDOR  = gold_path(PROJECT_ROOT, "dim_fornecedor")
FATO_CONTRATOS  = gold_path(PROJECT_ROOT, "fato_contratos")
FATO_SANCOES    = gold_path(PROJECT_ROOT, "fato_sancoes")
OUTPUT_PATH     = Path(serving_path(PROJECT_ROOT, "serving_fornecedor_perfil"))
OUTPUT_PATH_STR = serving_path(PROJECT_ROOT, "serving_fornecedor_perfil")
ensure_dir(OUTPUT_PATH.parent)

LOADED_AT = datetime.now(timezone.utc).isoformat()

# --- CNPJ Salt validation ---
load_dotenv(PROJECT_ROOT / ".env")
CNPJ_SALT = os.getenv("CNPJ_SALT")

if not CNPJ_SALT:
    raise EnvironmentError(
        "CNPJ_SALT not set. Add it to your .env file:\n"
        "  CNPJ_SALT=your_secret_salt_here\n"
        "Never commit the salt to Git."
    )

print(f"Output          : {OUTPUT_PATH}")
print(f"loaded_at       : {LOADED_AT}")
print(f"CNPJ_SALT       : {'*' * len(CNPJ_SALT)} (set, length={len(CNPJ_SALT)})")

## Step 2 — Load Gold sources as lazy views

In [ ]:
con = duckdb.connect()
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '8GB'")
con.execute("SET preserve_insertion_order = false")

con.execute(f"CREATE OR REPLACE VIEW v_dim     AS SELECT * FROM read_parquet('{DIM_FORNECEDOR}')")
con.execute(f"CREATE OR REPLACE VIEW v_fato_c  AS SELECT * FROM read_parquet('{FATO_CONTRATOS}')")
con.execute(f"CREATE OR REPLACE VIEW v_fato_s  AS SELECT * FROM read_parquet('{FATO_SANCOES}')")

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM v_dim    WHERE is_current = true) AS dim_current,
        (SELECT COUNT(*) FROM v_fato_c)                         AS fato_contratos,
        (SELECT COUNT(*) FROM v_fato_s)                         AS fato_sancoes
""").df()
print(counts.to_string(index=False))

## Step 3 — Compute cnpj_token mapping

### Why Python, not SQL
DuckDB has no native HMAC function. The token is computed in Python using
`hmac.new(salt, cnpj, sha256).hexdigest()` and loaded into DuckDB as a
native table for subsequent joins.

### Why materialise as a table
The mapping table is used twice — once in Step 6 to add cnpj_token to the
final output. Materialising avoids recomputing the HMAC for each join.

### Security boundary
`cnpj_normalized` appears in this step only as the input to HMAC.
After this step, only `cnpj_token` is used. `cnpj_normalized` is never
written to the output Parquet file.


In [ ]:
print("Computing cnpj_token mapping...")
t0 = datetime.now()

# Fetch all current CNPJs from dim_fornecedor
cnpjs = con.execute("""
    SELECT cnpj_normalized, supplier_sk
    FROM v_dim
    WHERE is_current = true
""").fetchall()

print(f"CNPJs to tokenise: {len(cnpjs):,}")

# Compute HMAC-SHA256 token for each CNPJ
def compute_token(cnpj: str, salt: str) -> str:
    return hmac.new(
        salt.encode("utf-8"),
        cnpj.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

token_rows = [
    (compute_token(cnpj, CNPJ_SALT), supplier_sk)
    for cnpj, supplier_sk in cnpjs
]

elapsed = (datetime.now() - t0).total_seconds()
print(f"Tokens computed  : {len(token_rows):,} in {elapsed:.1f}s")

# Load into DuckDB as a native table for join
con.execute("DROP TABLE IF EXISTS t_token_map")
con.execute("""
    CREATE TABLE t_token_map (
        cnpj_token  VARCHAR,
        supplier_sk VARCHAR
    )
""")
con.executemany("INSERT INTO t_token_map VALUES (?, ?)", token_rows)

# Verify uniqueness — collision would be a critical error
n_tokens   = con.execute("SELECT COUNT(*)           FROM t_token_map").fetchone()[0]
n_unique   = con.execute("SELECT COUNT(DISTINCT cnpj_token) FROM t_token_map").fetchone()[0]
n_sk_unique = con.execute("SELECT COUNT(DISTINCT supplier_sk) FROM t_token_map").fetchone()[0]

print(f"Total tokens     : {n_tokens:,}")
print(f"Unique tokens    : {n_unique:,}  {'OK' if n_unique == n_tokens else 'COLLISION DETECTED!'}")
print(f"Unique SK        : {n_sk_unique:,}  {'OK' if n_sk_unique == n_tokens else 'DUPLICATE SK!'}")

if n_unique != n_tokens:
    raise ValueError("HMAC collision detected — two different CNPJs produced the same token. Check salt.")

## Step 4 — Aggregate contract metrics per supplier

In [ ]:
print("Aggregating contract metrics...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_contratos_agg AS
    SELECT
        supplier_sk,
        COUNT(*)                                    AS total_contratos,
        SUM(valor_inicial)                          AS valor_total_contratos,
        AVG(valor_inicial)                          AS valor_medio_contrato,
        MAX(valor_inicial)                          AS maior_contrato,
        MIN(valor_inicial)                          AS menor_contrato,
        MIN(data_referencia)                        AS primeiro_contrato,
        MAX(data_referencia)                        AS ultimo_contrato,
        COUNT(DISTINCT codigo_orgao)
            + COUNT(DISTINCT cnpj_comprador)        AS orgaos_distintos,
        SUM(CASE WHEN _valor_negativo THEN 1 ELSE 0 END) AS contratos_valor_negativo
    FROM v_fato_c
    WHERE supplier_sk IS NOT NULL
    GROUP BY supplier_sk
""")

n       = con.execute("SELECT COUNT(*) FROM t_contratos_agg").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()
print(f"t_contratos_agg: {n:,} suppliers with contracts in {elapsed:.1f}s")

# Sanity: top suppliers by contract value
sample = con.execute("""
    SELECT supplier_sk, total_contratos, valor_total_contratos
    FROM t_contratos_agg
    ORDER BY valor_total_contratos DESC
    LIMIT 3
""").df()
print()
print(sample.to_string(index=False))

## Step 5 — Aggregate sanction metrics per supplier

In [ ]:
print("Aggregating sanction metrics...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_sancoes_agg AS
    SELECT
        supplier_sk,
        COUNT(*)                                            AS total_sancoes,
        SUM(CASE WHEN is_ativa THEN 1 ELSE 0 END)          AS total_sancoes_ativas,
        SUM(COALESCE(valor_multa, 0))                       AS valor_total_multas,
        MAX(data_inicio_sancao)                             AS ultima_sancao,
        COUNT(DISTINCT orgao_sancionador_nome)              AS orgaos_sancionadores_distintos
    FROM v_fato_s
    WHERE supplier_sk IS NOT NULL
    GROUP BY supplier_sk
""")

n       = con.execute("SELECT COUNT(*) FROM t_sancoes_agg").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()
print(f"t_sancoes_agg: {n:,} suppliers with sanctions in {elapsed:.1f}s")

## Step 6 — Build final serving table

Joins dim_fornecedor (current only) with aggregated metrics and cnpj_token.
`cnpj_normalized` is used only in the JOIN condition — never projected to output.


In [ ]:
print("Building serving_fornecedor_perfil...")
t0 = datetime.now()

con.execute(f"""
    CREATE OR REPLACE TABLE t_serving AS
    SELECT
        -- Identity: cnpj_token replaces cnpj_normalized (ADR-005)
        tk.cnpj_token,
        d.supplier_sk,

        -- Supplier attributes from dim_fornecedor (current version)
        d.razao_social,
        d.porte,
        d.situacao_cadastral,
        d.natureza_juridica_desc,
        d.uf,
        d.municipio_desc,
        d.tem_sancao,
        d.valid_from                                AS scd2_valid_from,

        -- Contract metrics (0 for suppliers with no contracts)
        COALESCE(c.total_contratos,            0)   AS total_contratos,
        COALESCE(c.valor_total_contratos,      0)   AS valor_total_contratos,
        COALESCE(c.valor_medio_contrato,       0)   AS valor_medio_contrato,
        COALESCE(c.maior_contrato,             0)   AS maior_contrato,
        COALESCE(c.menor_contrato,             0)   AS menor_contrato,
        c.primeiro_contrato,
        c.ultimo_contrato,
        COALESCE(c.orgaos_distintos,           0)   AS orgaos_distintos,
        COALESCE(c.contratos_valor_negativo,   0)   AS contratos_valor_negativo,

        -- Sanction metrics (0 for suppliers with no sanctions)
        COALESCE(s.total_sancoes,              0)   AS total_sancoes,
        COALESCE(s.total_sancoes_ativas,       0)   AS total_sancoes_ativas,
        COALESCE(s.valor_total_multas,         0)   AS valor_total_multas,
        s.ultima_sancao,
        COALESCE(s.orgaos_sancionadores_distintos, 0) AS orgaos_sancionadores_distintos,

        -- Audit
        '{LOADED_AT}'::TIMESTAMPTZ                  AS _loaded_at

    FROM v_dim d
    INNER JOIN t_token_map tk USING (supplier_sk)
    LEFT JOIN t_contratos_agg c USING (supplier_sk)
    LEFT JOIN t_sancoes_agg   s USING (supplier_sk)
    WHERE d.is_current = true
""")

n       = con.execute("SELECT COUNT(*) FROM t_serving").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()
print(f"t_serving: {n:,} rows in {elapsed:.1f}s")

## Step 7 — Write to Parquet

In [ ]:
con.execute(f"""
    COPY (SELECT * FROM t_serving)
    TO '{OUTPUT_PATH_STR}' (FORMAT PARQUET)
""")

# Cleanup staging tables
con.execute("DROP TABLE IF EXISTS t_token_map")
con.execute("DROP TABLE IF EXISTS t_contratos_agg")
con.execute("DROP TABLE IF EXISTS t_sancoes_agg")
con.execute("DROP TABLE IF EXISTS t_serving")

file_size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
print(f"Written : {OUTPUT_PATH}")
print(f"Size    : {file_size_mb:.2f} MB")

## Step 8 — Validation

In [ ]:
checks = []

total = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]

# CHECK 1 — row count = current suppliers in dim_fornecedor
expected = con.execute(
    "SELECT COUNT(*) FROM v_dim WHERE is_current = true"
).fetchone()[0]
checks.append(("Row count = current dim_fornecedor", total == expected,
               f"{total:,} (expected {expected:,})"))

# CHECK 2 — cnpj_token unique
unique_token = con.execute(
    f"SELECT COUNT(DISTINCT cnpj_token) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]
checks.append(("cnpj_token unique", unique_token == total,
               f"{unique_token:,} unique / {total:,} total"))

# CHECK 3 — cnpj_token not null
null_token = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE cnpj_token IS NULL"
).fetchone()[0]
checks.append(("cnpj_token not null", null_token == 0, f"{null_token} nulls"))

# CHECK 4 — cnpj_normalized NOT in output (ADR-005 compliance)
cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{OUTPUT_PATH_STR}')"
).df()["column_name"].tolist()
has_cnpj = "cnpj_normalized" in cols
checks.append(("cnpj_normalized absent (ADR-005)", not has_cnpj,
               "EXPOSED — ADR-005 VIOLATION" if has_cnpj else "Not present in output"))

# CHECK 5 — total_contratos >= 0 always
neg_contratos = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE total_contratos < 0
""").fetchone()[0]
checks.append(("total_contratos >= 0", neg_contratos == 0, f"{neg_contratos} negative"))

# CHECK 6 — total_sancoes_ativas <= total_sancoes always
invalid_sancoes = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE total_sancoes_ativas > total_sancoes
""").fetchone()[0]
checks.append(("total_sancoes_ativas <= total_sancoes", invalid_sancoes == 0,
               f"{invalid_sancoes} invalid rows"))

# CHECK 7 — tem_sancao=True aligns with total_sancoes > 0
# Note: tem_sancao comes from dim_fornecedor (computed at load time)
# total_sancoes comes from fato_sancoes (may differ slightly — 73 CNPJs not in dim)
mismatch = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE tem_sancao = true AND total_sancoes = 0
""").fetchone()[0]
checks.append(("tem_sancao=True implies total_sancoes > 0", mismatch == 0,
               f"{mismatch} mismatches"))

# CHECK 8 — suppliers with contracts have primeiro_contrato not null
null_first = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE total_contratos > 0 AND primeiro_contrato IS NULL
""").fetchone()[0]
checks.append(("primeiro_contrato not null when has contracts", null_first == 0,
               f"{null_first} nulls"))

# ── Report ────────────────────────────────────────────────────────────────────
print(f"\n{'CHECK':<55} {'STATUS':<8} DETAIL")
print("-" * 90)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<55} [{status}]   {detail}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks PASSED — serving_fornecedor_perfil ready.")
else:
    print("One or more checks FAILED — review output above.")

# ── Distribution stats ────────────────────────────────────────────────────────
print()
print("Contract coverage:")
cov = con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE total_contratos > 0)  AS com_contratos,
        COUNT(*) FILTER (WHERE total_contratos = 0)  AS sem_contratos,
        COUNT(*)                                      AS total
    FROM read_parquet('{OUTPUT_PATH_STR}')
""").df()
print(cov.to_string(index=False))

print()
print("Sanction coverage:")
san = con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE total_sancoes > 0)        AS com_sancao,
        COUNT(*) FILTER (WHERE total_sancoes_ativas > 0) AS com_sancao_ativa,
        COUNT(*) FILTER (WHERE total_sancoes = 0)        AS sem_sancao,
        COUNT(*)                                         AS total
    FROM read_parquet('{OUTPUT_PATH_STR}')
""").df()
print(san.to_string(index=False))

print()
print("Sample rows (com_sancao_ativa=True):")
sample = con.execute(f"""
    SELECT cnpj_token, razao_social, porte, uf,
           total_contratos, valor_total_contratos,
           total_sancoes, total_sancoes_ativas, valor_total_multas
    FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE total_sancoes_ativas > 0
    ORDER BY valor_total_contratos DESC
    LIMIT 4
""").df()
print(sample.to_string(index=False))